In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import RepeatedStratifiedKFold
from xgboost import XGBClassifier, XGBRegressor

from fraud_detection import data_loader, metrics

datapath = "../data/transformed_label_and_damage.parquet"

In [ ]:
seed = 42
np.random.seed(seed)


In [ ]:
def cost_fn(probs, damage, cost_fp, tain_tp=5):
    return probs > cost_fp / (tain_tp + cost_fp + damage)

def id(X, y):
    return X, y

In [ ]:
X, _ = data_loader.load_data(datapath, drop_features=data_loader.useless_features)
for i, c in enumerate(X.columns):
    if "diff" in c:
      print(f"Feature {i}: {c}")

In [ ]:
X, targets = data_loader.load_data_np(datapath, drop_features=data_loader.useless_features)

In [ ]:
def fit_regressor_and_predict(X_train, y_train, X_test, y_test, traindata_function: callable):
    X_train, y_train = traindata_function(X_train, y_train)

    model = XGBRegressor(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        objective="reg:squarederror",
    )

    #sample_weights = np.where(y_train[:, 0] > 0, 200.0, 1.0)
    model.fit(X_train, y_train[:, 1])#, sample_weight=sample_weights)

    #model.fit(X_train, y_train[:, 1])
    preds = model.predict(X_test)
    bew = metrics.regression(preds, y_test[:, 1])

    return model, bew, preds

In [ ]:
def eval_rabattfaelle(X, y, preds):
    # F?lle mit Rabatt
    idxr = np.where(X[:, 28] > 0)[0]
    # Fraudf?lle
    idxf = np.where(y[:, 0] == 1)[0]
    # F?lle mit Rabatt und Fraud
    idxrf = np.intersect1d(idxr, idxf)

    # Now we want the indices of idxrf where preds == 1 (i.e. correctly predicted fraud)
    idxrf_pred_mask = preds[idxrf] == 1
    idxrf_pred = idxrf[idxrf_pred_mask]

    # Anzahl der F?lle mit Rabatt und Fraud
    num_rf = len(idxrf)
    # Anzahl der korrekt vorhergesagten F?lle mit Rabatt und Fraud
    num_rf_pred = len(idxrf_pred)

    # Schaden der Fraud-F?lle mit Rabatt
    damage_rf = y[idxrf, 1]
    # Schaden der korrekt vorhergesagten Fraud-F?lle mit Rabatt
    damage_rf_pred = y[idxrf_pred, 1]

    assert damage_rf_pred.min() > 0, "Damage of predicted fraud cases with discount should be greater than 0"
    assert damage_rf.min() > 0, "Damage of fraud cases with discount should be greater than 0"

    return {
        "Anzahl F?lle mit Rabatt und Fraud": num_rf,
        "Anzahl korrekt vorhergesagter F?lle": num_rf_pred,
        "Schaden der F?lle mit Rabatt und Fraud": damage_rf,
        "Schaden der korrekt vorhergesagten F?lle": damage_rf_pred,
    }

In [ ]:
def create_and_evaluate_model(X_train, y_train, X_test, y_test, train_data_function: callable):
    threshold = 0.0
    clf = XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        objective="binary:logistic",
    )
    clf.fit(X_train, y_train[:, 0])
    probs_baseline = clf.predict_proba(X_test)[:, 1]
    preds_baseline = clf.predict(X_test)

    metrics_baseline = metrics.bewertung(probs_baseline, preds_baseline, y_test[:, 0], y_test[:, 1])

    model, bew, preds = fit_regressor_and_predict(X_train, y_train, X_test, y_test, train_data_function)

    damage_preds = model.predict(X_test)

    preds = (probs_baseline > threshold) & cost_fn(probs_baseline, damage_preds, 10)

    res = eval_rabattfaelle(X_test, y_test, preds)
    return res


In [ ]:
def run_experiment(X, targets, train_data_function, n_splits=5, n_repeats=1, random_state=42):

    # Initialize the RepeatedStratifiedKFold
    skf = RepeatedStratifiedKFold(
        n_splits=n_splits, n_repeats=n_repeats, random_state=random_state
    )


    results = []
    for i, (train_idx, test_idx) in enumerate(skf.split(X, targets[:, 0])):
        print(f"Fold {i + 1}/{n_splits * n_repeats}")
        res = create_and_evaluate_model(
            X[train_idx],
            targets[train_idx],
            X[test_idx],
            targets[test_idx],
            train_data_function,
        )
        results.append(res)

    return results


In [ ]:
n_splits = 5
n_repeats = 10

results = run_experiment(X, targets, id, n_splits=n_splits, n_repeats=n_repeats, random_state=seed)

In [ ]:
k0, k1 = list(results[0].keys())[:2]
d1 = pd.DataFrame({
  k0: [d[k0] for d in results],
  k1: [d[k1] for d in results],
})

In [ ]:
d1["Anteil erkannt"] = d1[k1] / d1[k0]
d1

In [ ]:
d1.describe().T["mean"].round(2)

In [ ]:
k2, k3 = list(results[0].keys())[2:4]
d2 = pd.DataFrame({
  k2: [d[k2].sum() for d in results],
  k3: [d[k3].sum() for d in results],
})
d2["Anteil Schaden verhindert"] = d2[k3] / d2[k2]
d2

In [ ]:
d2.describe().T["mean"].round(2)